# 03 — PINN Implementation

Physics-Informed Neural Network solver for all four PDEs.

**Architecture spec** (from problem statement):
- 4 hidden layers × 64 units, **tanh** activation (never ReLU — second derivatives would be zero)
- 10,000 collocation points interior, 200 IC points, 200 BC points
- Adam at 1e-3 for ~20k steps, optional L-BFGS polish

**Key implementation note**: autodiff gives u_t, u_x, u_xx with respect to
*network inputs* using `torch.autograd.grad(..., create_graph=True)`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PINN using device: {device}')

In [ ]:
# ============================================================
# Network Architecture
# ============================================================

class PINN(nn.Module):
    """
    Standard MLP for PINN tasks.

    Spec: 4 hidden layers x 64 units, Tanh activation.
    Input dimension is flexible (2 for heat/advection, 3 for water waves/oscillator).

    CRITICAL: Use tanh, not ReLU. The physics loss includes second derivatives
    of the output w.r.t. inputs. ReLU has zero second derivative everywhere,
    making the residual term silently meaningless.
    """

    def __init__(self, in_dim=2, out_dim=1, hidden_dim=64, n_layers=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, *inputs):
        """Accepts 2+ tensors (e.g. x, t or x, z, t) and concatenates them."""
        return self.net(torch.cat(inputs, dim=-1))


# ============================================================
# Autodiff gradient helper
# ============================================================

def grad(y, x):
    """Compute dy/dx via autodiff. create_graph=True allows higher-order derivatives."""
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True
    )[0]


print(f'PINN architecture ready. Parameter count: {sum(p.numel() for p in PINN().parameters())}')

In [ ]:
# ============================================================
# Exact Initial / Boundary Conditions
# (copied from 01_exact_solutions for standalone use)
# ============================================================

def heat_exact_ic(x, k_max=16, seed=42):
    rng = torch.Generator()
    rng.manual_seed(seed)
    x = x.view(-1, 1) if x.dim() == 1 else x
    u = torch.zeros_like(x, dtype=torch.float32)
    for k in range(1, k_max + 1):
        a_k = torch.randn(1, generator=rng).item() * (k ** -2)
        b_k = torch.randn(1, generator=rng).item() * (k ** -2)
        u = u + a_k * torch.sin(2*math.pi*k*x) + b_k * torch.cos(2*math.pi*k*x)
    return u

In [ ]:
# ============================================================
# PDE-specific loss functions
# ============================================================

def heat_losses(model, alpha=0.01, n_f=10000, n_bc=200):
    """Losses for u_t = alpha * u_xx on x in [0,1], t in [0,1]."""
    # Interior collocation
    x_f = torch.rand(n_f, 1, requires_grad=True, device=device)
    t_f = torch.rand(n_f, 1, requires_grad=True, device=device)
    u   = model(x_f, t_f)
    u_t = grad(u, t_f)
    u_x = grad(u, x_f)
    u_xx = grad(u_x, x_f)
    loss_res = torch.mean((u_t - alpha * u_xx) ** 2)

    # Initial condition: u(x,0) = u0(x)
    x_ic = torch.rand(n_bc, 1, requires_grad=True, device=device)
    t_ic = torch.zeros(n_bc, 1, requires_grad=True, device=device)
    loss_ic = torch.mean((model(x_ic, t_ic) - heat_exact_ic(x_ic)) ** 2)

    # Periodic BC: u(0,t) = u(1,t) and u_x(0,t) = u_x(1,t)
    t_bc = torch.rand(n_bc, 1, requires_grad=True, device=device)
    x_left  = torch.zeros(n_bc, 1, requires_grad=True, device=device)
    x_right = torch.ones( n_bc, 1, requires_grad=True, device=device)
    u_left  = model(x_left,  t_bc)
    u_right = model(x_right, t_bc)
    loss_bc = (torch.mean((u_left - u_right) ** 2) +
               torch.mean((grad(u_left, x_left) - grad(u_right, x_right)) ** 2))

    return loss_res, loss_ic, loss_bc


def advection_losses(model, c=1.0, n_f=10000, n_bc=200):
    """Losses for u_t + c * u_x = 0."""
    x_f = torch.rand(n_f, 1, requires_grad=True, device=device)
    t_f = torch.rand(n_f, 1, requires_grad=True, device=device)
    u   = model(x_f, t_f)
    loss_res = torch.mean((grad(u, t_f) + c * grad(u, x_f)) ** 2)

    x_ic = torch.rand(n_bc, 1, requires_grad=True, device=device)
    t_ic = torch.zeros(n_bc, 1, requires_grad=True, device=device)
    loss_ic = torch.mean((model(x_ic, t_ic) - heat_exact_ic(x_ic)) ** 2)

    # Periodic BC
    t_bc = torch.rand(n_bc, 1, requires_grad=True, device=device)
    x_left  = torch.zeros(n_bc, 1, requires_grad=True, device=device)
    x_right = torch.ones( n_bc, 1, requires_grad=True, device=device)
    loss_bc = torch.mean((model(x_left, t_bc) - model(x_right, t_bc)) ** 2)

    return loss_res, loss_ic, loss_bc


def water_wave_losses(model, g=9.81, h=0.159, L=1.0, T_max=2.0, n_f=10000, n_bc=200):
    """Losses for phi_xx + phi_zz = 0 with seabed and free-surface BCs."""
    # Interior: Laplace equation
    x_f = (torch.rand(n_f, 1) * L).requires_grad_(True).to(device)
    z_f = (torch.rand(n_f, 1) * -h).requires_grad_(True).to(device)
    t_f = (torch.rand(n_f, 1) * T_max).requires_grad_(True).to(device)
    phi = model(x_f, z_f, t_f)
    loss_res = torch.mean((grad(grad(phi, x_f), x_f) + grad(grad(phi, z_f), z_f)) ** 2)

    # IC: phi(x,z,0) = exact
    k = 2 * math.pi / L
    omega = math.sqrt(g * k * math.tanh(k * h))
    x_ic = (torch.rand(n_bc, 1) * L).requires_grad_(True).to(device)
    z_ic = (torch.rand(n_bc, 1) * -h).requires_grad_(True).to(device)
    t_ic = torch.zeros(n_bc, 1, requires_grad=True, device=device)
    a = 0.01
    phi_ic_exact = (a * g / omega) * (torch.cosh(k * (z_ic + h)) / math.cosh(k * h)) * torch.sin(k * x_ic)
    loss_ic = torch.mean((model(x_ic, z_ic, t_ic) - phi_ic_exact) ** 2)

    # BC seabed: phi_z = 0 at z = -h
    x_bc = (torch.rand(n_bc, 1) * L).requires_grad_(True).to(device)
    t_bc = (torch.rand(n_bc, 1) * T_max).requires_grad_(True).to(device)
    z_bed = (torch.ones(n_bc, 1) * -h).requires_grad_(True).to(device)
    loss_seabed = torch.mean(grad(model(x_bc, z_bed, t_bc), z_bed) ** 2)

    # BC free surface: phi_tt + g * phi_z = 0 at z = 0
    z_surf = torch.zeros(n_bc, 1, requires_grad=True, device=device)
    phi_surf = model(x_bc, z_surf, t_bc)
    phi_t    = grad(phi_surf, t_bc)
    phi_tt   = grad(phi_t, t_bc)
    phi_z    = grad(phi_surf, z_surf)
    loss_surf = torch.mean((phi_tt + g * phi_z) ** 2)

    return loss_res, loss_ic, loss_seabed + loss_surf


def oscillator_losses(model, omega0=2.0, zeta=0.1, n_f=10000, n_bc=200):
    """Losses for z_tt + 2*zeta*omega0*z_t + omega0^2*z = 0.
    
    The PINN takes (t, z0, v0) -> z, so it learns a FAMILY of trajectories
    parametrized by initial conditions.
    """
    t_f  = (torch.rand(n_f, 1) * 5.0).requires_grad_(True).to(device)
    z0_f = (2 * torch.rand(n_f, 1) - 1).to(device)  # random IC in [-1, 1]
    v0_f = (2 * torch.rand(n_f, 1) - 1).to(device)
    z    = model(t_f, z0_f, v0_f)
    z_t  = grad(z, t_f)
    z_tt = grad(z_t, t_f)
    loss_res = torch.mean((z_tt + 2*zeta*omega0*z_t + omega0**2 * z) ** 2)

    # IC: z(0, z0, v0) = z0, z_t(0, z0, v0) = v0
    t_ic  = torch.zeros(n_bc, 1, requires_grad=True, device=device)
    z0_ic = (2 * torch.rand(n_bc, 1) - 1).to(device)
    v0_ic = (2 * torch.rand(n_bc, 1) - 1).to(device)
    z_ic  = model(t_ic, z0_ic, v0_ic)
    z_t_ic = grad(z_ic, t_ic)
    loss_ic = torch.mean((z_ic - z0_ic) ** 2) + torch.mean((z_t_ic - v0_ic) ** 2)

    return loss_res, loss_ic, torch.tensor(0.0, device=device)

In [ ]:
# ============================================================
# Training loop
# ============================================================

def train_pinn(
    pde_type,
    epochs=20000,
    lr=1e-3,
    log_every=2000,
    weight_res=1.0,
    weight_ic=1.0,
    weight_bc=1.0,
    lbfgs_steps=0,
    **kwargs
):
    """
    Train a PINN for a specified PDE.

    Args:
        pde_type: 'heat' | 'advection' | 'water_waves' | 'oscillator'
        epochs: Adam steps
        lr: Adam learning rate
        log_every: print interval
        weight_res/ic/bc: loss term weights (default all 1)
        lbfgs_steps: optional L-BFGS polish steps after Adam
        **kwargs: passed to PDE loss function (e.g. alpha=0.01, c=20.0)

    Returns:
        model: trained PINN
        history: list of (epoch, res, ic, bc, total) tuples
    """
    pde_fn_map = {
        'heat':        (heat_losses,       2),
        'advection':   (advection_losses,   2),
        'water_waves': (water_wave_losses,  3),
        'oscillator':  (oscillator_losses,  3),
    }
    assert pde_type in pde_fn_map, f'Unknown PDE: {pde_type}'
    loss_fn, in_dim = pde_fn_map[pde_type]

    model = PINN(in_dim=in_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = []

    print(f'\n{"="*55}')
    print(f' Training PINN: {pde_type.upper()}  ({epochs} Adam steps)')
    print(f'{"="*55}')

    t_start = time.time()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        l_res, l_ic, l_bc = loss_fn(model, **kwargs)
        total = weight_res * l_res + weight_ic * l_ic + weight_bc * l_bc
        total.backward()
        optimizer.step()

        if epoch % log_every == 0 or epoch == 1:
            r, i, b, tot = l_res.item(), l_ic.item(), l_bc.item(), total.item()
            history.append((epoch, r, i, b, tot))
            print(f'  Epoch {epoch:06d} | Res={r:.3e} | IC={i:.3e} | BC={b:.3e} | Total={tot:.3e}')

    # Optional L-BFGS polish
    if lbfgs_steps > 0:
        print(f'\nApplying L-BFGS polish ({lbfgs_steps} steps)...')
        lbfgs = optim.LBFGS(model.parameters(), max_iter=lbfgs_steps)
        def closure():
            lbfgs.zero_grad()
            l_r, l_i, l_b = loss_fn(model, **kwargs)
            loss = weight_res*l_r + weight_ic*l_i + weight_bc*l_b
            loss.backward()
            return loss
        lbfgs.step(closure)

    elapsed = time.time() - t_start
    print(f'\nTraining complete in {elapsed:.1f}s')
    return model, history


def plot_loss_history(history, title='PINN Training Loss'):
    """Plot residual, IC, BC, and total loss curves."""
    epochs = [h[0] for h in history]
    plt.figure(figsize=(10, 4))
    for idx, label in [(1, 'Residual'), (2, 'IC'), (3, 'BC'), (4, 'Total')]:
        vals = [h[idx] for h in history]
        plt.semilogy(epochs, vals, label=label)
    plt.title(title); plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, which='both', alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# DEMO: Heat Equation (positive control)
# ============================================================
# This should converge cleanly. If it doesn't, the architecture is broken.

print('=== Heat Equation (Positive Control) ===')
model_heat, hist_heat = train_pinn(
    'heat', epochs=10000, alpha=0.01, log_every=2000
)
plot_loss_history(hist_heat, 'Heat PINN Training Loss')

In [ ]:
# ============================================================
# Evaluate heat PINN vs exact solution
# ============================================================

def heat_exact_solution_eval(x, t, alpha=0.01, k_max=16, seed=42):
    rng = torch.Generator(); rng.manual_seed(seed)
    x = x.view(-1, 1)
    u = torch.zeros_like(x)
    for k in range(1, k_max + 1):
        a_k = torch.randn(1, generator=rng).item() * k**-2
        b_k = torch.randn(1, generator=rng).item() * k**-2
        decay = math.exp(-alpha * (2*math.pi*k)**2 * t)
        u = u + (a_k * torch.sin(2*math.pi*k*x) + b_k * torch.cos(2*math.pi*k*x)) * decay
    return u

model_heat.eval()
x_test = torch.linspace(0, 1, 256, device=device).view(-1, 1)
t_test_val = 1.0
t_test = torch.full_like(x_test, t_test_val)

with torch.no_grad():
    u_pred_heat = model_heat(x_test, t_test)

u_exact_heat = heat_exact_solution_eval(x_test.cpu(), t_test_val)

# Relative L2 error
err = (torch.norm(u_pred_heat.cpu() - u_exact_heat) /
       (torch.norm(u_exact_heat) + 1e-8)).item()
print(f'Heat PINN Relative L2 Error at t={t_test_val}: {err:.6f}')

plt.figure(figsize=(10, 4))
plt.plot(x_test.cpu().numpy(), u_exact_heat.numpy(), label='Exact', linewidth=2)
plt.plot(x_test.cpu().numpy(), u_pred_heat.cpu().numpy(), label='PINN', linestyle='--', linewidth=2)
plt.title(f'Heat PINN vs Exact (t={t_test_val}, L2={err:.4f})')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Advection at c=1 (easy) and c=20 (hard)
# ============================================================

print('=== Advection c=1.0 (Sanity Check) ===')
model_adv_c1, hist_c1 = train_pinn('advection', epochs=10000, c=1.0, log_every=2000)

print('\n=== Advection c=20.0 (The Cliff — expect failure) ===')
model_adv_c20, hist_c20 = train_pinn('advection', epochs=10000, c=20.0, log_every=2000)

In [ ]:
# ============================================================
# Damped Oscillator
# ============================================================

print('=== Damped Oscillator ===')
model_osc, hist_osc = train_pinn('oscillator', epochs=10000, omega0=2.0, zeta=0.1, log_every=2000)
plot_loss_history(hist_osc, 'Oscillator PINN Training Loss')

In [ ]:
# Evaluate oscillator PINN vs exact
def oscillator_exact_sol(t, z0=1.0, v0=0.0, omega0=2.0, zeta=0.1):
    omega_d = omega0 * math.sqrt(1 - zeta**2)
    C1, C2 = z0, (v0 + zeta*omega0*z0) / omega_d
    return torch.exp(-zeta*omega0*t) * (C1*torch.cos(omega_d*t) + C2*torch.sin(omega_d*t))

model_osc.eval()
t_eval = torch.linspace(0, 10, 400, device=device).view(-1, 1)
z0_eval = torch.ones_like(t_eval)   # z0=1
v0_eval = torch.zeros_like(t_eval)  # v0=0

with torch.no_grad():
    z_pred = model_osc(t_eval, z0_eval, v0_eval)

z_exact = oscillator_exact_sol(t_eval.cpu())
err_osc = (torch.norm(z_pred.cpu() - z_exact) / (torch.norm(z_exact) + 1e-8)).item()
print(f'Oscillator PINN Relative L2 Error: {err_osc:.6f}')

plt.figure(figsize=(10, 4))
plt.plot(t_eval.cpu().numpy(), z_exact.numpy(), label='Exact', linewidth=2)
plt.plot(t_eval.cpu().numpy(), z_pred.cpu().numpy(), label='PINN', linestyle='--', linewidth=2)
plt.axvline(x=5, color='red', linestyle=':', label='Train boundary')
plt.title(f'Oscillator PINN vs Exact (L2={err_osc:.4f})')
plt.xlabel('t'); plt.ylabel('z(t)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()